# Stage 3 · AR verifiers (Qwen, Mistral, Llama, Gemma, small models)

Runtime: **A100** (40 GB; the FP32 checks need it). One fresh runtime for this notebook only.
Llama, Gemma and Mistral are gated: accept their licenses on Hugging Face first, and add your token as the Colab secret `HF_TOKEN`.

What it runs (PLAN.md §4 and §7):
1. The AR reference check for Qwen and Mistral: the rendered prompts, our one-pass readout against the first step of the model's own generate(), and the same items in FP32 against BF16 (the check that would have caught the eager-attention bug). This is the evidence for signing off Mistral's sanity failure.
2. Qwen and Mistral on the 71 development pairs: a pipeline check (real tokenizer lengths, usable-scores gate), not a result.
3. Sanity check plus reference check for Llama-3.1-8B and Gemma-2-9B (Gemma with eager attention: SDPA skips its soft-capping).
4. Sanity check plus reference check for Llama-3.2-3B (gate A's small AR verifier) and Qwen2.5-1.5B.

In [ ]:
BRANCH = "main"   # the branch to test; main once the stage-3 PR is merged
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r requirements.txt -r requirements-analysis.txt
# Add a Hugging Face token as a Colab secret named HF_TOKEN (key icon on the left) with access
# to Llama, Gemma and Mistral; this cell then logs in before anything downloads.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
OUT = "results/stage3/ar"

## 1. Reference check: Qwen and Mistral

In [ ]:
!python -m harness.ar_reference --config configs/ar_pair_sanity.json --input bank/sanity/sanity_set.jsonl --output {OUT}/reference_qwen_mistral.json --fp32-check

## 2. Qwen and Mistral on the development pairs (pipeline check)

In [ ]:
!python -m harness.run --config configs/ar_pair_dev.json --input bank/dev/dev_bank.jsonl --output-dir {OUT}/dev-qwen-mistral --device cuda:0 --dtype bf16 --no-load-in-4bit
!python -m harness.evaluate --bank bank/dev/dev_bank.jsonl --labels bank/dev/dev_labels.jsonl --run-dir {OUT}/dev-qwen-mistral --output-dir {OUT}/dev-qwen-mistral-analysis

## 3. Llama-3.1-8B and Gemma-2-9B: sanity check and reference check

In [ ]:
!python -m harness.run --config configs/ar_sanity_llama_gemma.json --input bank/sanity/sanity_set.jsonl --output-dir {OUT}/sanity-llama-gemma --device cuda:0 --dtype bf16 --no-load-in-4bit
!python -m harness.evaluate --sanity --bank bank/sanity/sanity_set.jsonl --labels bank/sanity/sanity_set.jsonl --run-dir {OUT}/sanity-llama-gemma --output-dir {OUT}/sanity-llama-gemma-analysis
!python -m harness.ar_reference --config configs/ar_sanity_llama_gemma.json --input bank/sanity/sanity_set.jsonl --output {OUT}/reference_llama_gemma.json

## 4. Llama-3.2-3B and Qwen2.5-1.5B: sanity check and reference check

In [ ]:
!python -m harness.run --config configs/ar_sanity_small.json --input bank/sanity/sanity_set.jsonl --output-dir {OUT}/sanity-small --device cuda:0 --dtype bf16 --no-load-in-4bit
!python -m harness.evaluate --sanity --bank bank/sanity/sanity_set.jsonl --labels bank/sanity/sanity_set.jsonl --run-dir {OUT}/sanity-small --output-dir {OUT}/sanity-small-analysis
!python -m harness.ar_reference --config configs/ar_sanity_small.json --input bank/sanity/sanity_set.jsonl --output {OUT}/reference_small.json --fp32-check

A sanity evaluation that fails the 0.90 bar exits with code 1 and prints an error line; that is the result, not a crash. Keep going.

## Download the results
This zips `results/stage3/ar/` (scores, manifests and reports). Send the zip back, or unzip it into the repo at the same path.

In [ ]:
!cd /content/project && zip -qr /content/stage3_ar.zip results/stage3/ar
from google.colab import files
files.download("/content/stage3_ar.zip")